# Validation Complète Modèle 3 — Orientation des composants (YOLO11n-OBB) — Maximum de Métriques

Ce notebook exécute la validation complète (`model.val(...)`) sur votre modèle de détection orientée entraîné (`best_ep100.pt` / `best_ep70.pt`) et extrait **le maximum de métriques OBB ainsi que les graphiques** :
- **Bloc 1** : Configuration des chemins (`MODEL_PATH`, `DATA_YAML`, split `val` ou `test`, paramètres d'évaluation).
- **Bloc 2** : Exécution de `model.val(..., task="obb")` avec génération des courbes et matrices (`plots=True`, `save_json=True`).
- **Bloc 3** : Extraction détaillée de toutes les métriques globales et par classe pour **OBB** (Précision, Rappel, F1-score, `mAP50`, `mAP75`, `mAP50-95`, fitness, temps d'inférence/FPS, matrice de confusion, métriques par image).
- **Bloc 4** : Affichage des courbes générées (Matrices de confusion OBB, PR-curve, F1-curve, P-curve, R-curve) et export en CSV / JSON.
- **Bloc 5** : Comparaison multi-paliers (`best_ep10.pt` à `best_ep100.pt`) sur le split choisi (`val` ou `test`).

## Bloc 1 — Configuration des chemins et paramètres de validation

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import torch
from ultralytics import YOLO

# 1. Chemin vers votre modèle entraîné du Module 3 (Orientation YOLO11n-OBB)
# Note : d'après vos logs d'entraînement, best_ep70.pt atteint 0.489 de mAP50-95 et best_ep100.pt 0.478.
MODEL_PATH = Path(r"D:\S5\Projet_d_expertise\projet\models\03_orientation_des_composants_paliers\best_ep100.pt").resolve()

# 2. Chemin vers le fichier data.yaml correspondant au dataset OBB
DATA_YAML = Path(r"D:\S5\Projet_d_expertise\projet\data\processed\wiring harness connectors segmen.v1i.yolov8-obb\data.yaml").resolve()

# 3. Split à évaluer ('val' ou 'test')
EVAL_SPLIT = "val"

# 4. Paramètres de validation (cohérents avec l'entraînement : imgsz=640, batch=16, iou=0.7)
IMG_SIZE = 640
BATCH_SIZE = 16
IOU_THRESHOLD = 0.7
# Laisser CONF_THRESHOLD = None pour le calcul standard mAP (recommandé), ou fixer ex. 0.25 pour matrice opérationnelle
CONF_THRESHOLD = None

OUTPUT_DIR = Path(r"D:\S5\Projet_d_expertise\projet\evaluations").resolve()
DEVICE = 0 if torch.cuda.is_available() else "cpu"

print("Configuration de validation (Module 3 - Orientation OBB) :")
print("  - Modèle   :", MODEL_PATH)
print("  - Dataset  :", DATA_YAML)
print("  - Split    :", EVAL_SPLIT)
print("  - Device   :", DEVICE)

Configuration de validation (Module 3 - Orientation OBB) :
  - Modèle   : D:\S5\Projet_d_expertise\projet\models\03_orientation_des_composants_paliers\best_ep100.pt
  - Dataset  : D:\S5\Projet_d_expertise\projet\data\processed\wiring harness connectors segmen.v1i.yolov8-obb\data.yaml
  - Split    : val
  - Device   : cpu


## Bloc 2 — Lancement de la validation complète (`model.val`)

In [2]:
# Recherche automatique du dernier modèle OBB si le chemin par défaut n'existe pas
if not MODEL_PATH.exists():
    found_models = sorted(
        Path(r"D:\S5\Projet_d_expertise\projet\models").glob("*03_orientation_des_composants*/**/*.pt"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if found_models:
        MODEL_PATH = found_models[0].resolve()
        print(f"Modèle trouvé automatiquement : {MODEL_PATH}")
    else:
        raise FileNotFoundError(f"Modèle introuvable : {MODEL_PATH}")

if not DATA_YAML.exists():
    raise FileNotFoundError(f"Fichier data.yaml introuvable : {DATA_YAML}")

model = YOLO(str(MODEL_PATH))
print(f"Tâche du modèle : {model.task} | Classes : {model.names}")

run_eval_name = f"03_orientation_des_composants_{MODEL_PATH.stem}_{EVAL_SPLIT}"
val_kwargs = dict(
    data=str(DATA_YAML),
    task="obb",
    split=EVAL_SPLIT,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    iou=IOU_THRESHOLD,
    device=DEVICE,
    plots=True,
    save_json=True,
    project=str(OUTPUT_DIR),
    name=run_eval_name,
    exist_ok=True,
)
if CONF_THRESHOLD is not None:
    val_kwargs["conf"] = CONF_THRESHOLD

metrics = model.val(**val_kwargs)
save_dir = Path(metrics.save_dir)
print(f"Validation terminée. Résultats sauvegardés dans : {save_dir}")

Tâche du modèle : obb | Classes : {0: 'cable', 1: 'cable-tie', 2: 'clip', 3: 'connector', 4: 'label'}
Ultralytics 8.4.26  Python-3.12.7 torch-2.11.0+cpu CPU (Intel Core i5-10210U 1.60GHz)
YOLO11n-obb summary (fused): 110 layers, 2,654,698 parameters, 0 gradients, 6.6 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 66.623.2 MB/s, size: 38.0 KB)
val: Scanning D:\S5\Projet_d_expertise\projet\data\processed\wiring harness connectors segmen.v1i.yolov8-obb\valid\labels.cache... 22 images, 4 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 22/22  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 3.8s/it 7.6s<16.1s
                   all         22         88      0.885       0.53      0.619      0.478
                 cable          3         16          1     0.0669      0.271      0.103
             cable-tie          2          2      0.591          1      0.995      0.849
                  clip          1          5       

## Bloc 3 — Extraction de toutes les métriques OBB (Globales, par classe, par image, vitesse & confusion)

In [3]:
# 1. Dictionnaire brut de toutes les métriques scalaires Ultralytics
results_dict = getattr(metrics, "results_dict", {})
df_global = pd.DataFrame([
    {"Metrique": k, "Valeur": round(float(v), 6)} for k, v in results_dict.items()
])

# 2. Extraction détaillée des métriques OBB (Ultralytics stocke les scores OBB dans metrics.obb ou metrics.box)
obb_obj = getattr(metrics, "obb", None) or getattr(metrics, "box", None)
extra_rows = []
if obb_obj is not None and hasattr(obb_obj, "map"):
    p_mean = float(getattr(obb_obj, "mp", np.nan))
    r_mean = float(getattr(obb_obj, "mr", np.nan))
    f1_mean = (2 * p_mean * r_mean / (p_mean + r_mean)) if (p_mean + r_mean) > 0 else 0.0
    extra_rows.extend([
        {"Metrique": "OBB/Precision_moyenne (mp)", "Valeur": round(p_mean, 6)},
        {"Metrique": "OBB/Recall_moyen (mr)", "Valeur": round(r_mean, 6)},
        {"Metrique": "OBB/F1_moyen", "Valeur": round(f1_mean, 6)},
        {"Metrique": "OBB/mAP50", "Valeur": round(float(getattr(obb_obj, "map50", np.nan)), 6)},
        {"Metrique": "OBB/mAP75", "Valeur": round(float(getattr(obb_obj, "map75", np.nan)), 6)},
        {"Metrique": "OBB/mAP50-95", "Valeur": round(float(getattr(obb_obj, "map", np.nan)), 6)},
    ])

# 3. Vitesse d'exécution (ms / image)
speed = getattr(metrics, "speed", {})
for sp_k, sp_v in speed.items():
    extra_rows.append({"Metrique": f"Speed_{sp_k}_ms", "Valeur": round(float(sp_v), 3)})
if speed:
    total_ms = sum(float(v) for v in speed.values())
    extra_rows.append({"Metrique": "Speed_total_ms", "Valeur": round(total_ms, 3)})
    if total_ms > 0:
        extra_rows.append({"Metrique": "FPS_estime", "Valeur": round(1000.0 / total_ms, 2)})

df_global = pd.concat([df_global, pd.DataFrame(extra_rows)], ignore_index=True).drop_duplicates(subset=["Metrique"])
print("=== MÉTRIQUES GLOBALES (OBB) ===")
display(df_global)

# 4. Métriques détaillées PAR CLASSE
per_class_rows = []
if obb_obj is not None and hasattr(obb_obj, "ap_class_index"):
    class_indices = obb_obj.ap_class_index
    for idx_pos, cid in enumerate(class_indices):
        cid_int = int(cid)
        cname = model.names.get(cid_int, str(cid_int))
        obb_p = float(obb_obj.p[idx_pos]) if idx_pos < len(obb_obj.p) else np.nan
        obb_r = float(obb_obj.r[idx_pos]) if idx_pos < len(obb_obj.r) else np.nan
        obb_f1 = float(obb_obj.f1[idx_pos]) if (hasattr(obb_obj, "f1") and idx_pos < len(obb_obj.f1)) else (
            (2 * obb_p * obb_r / (obb_p + obb_r)) if (obb_p + obb_r) > 0 else 0.0
        )
        obb_ap50 = float(obb_obj.ap50[idx_pos]) if idx_pos < len(obb_obj.ap50) else np.nan
        obb_ap = float(obb_obj.ap[idx_pos]) if idx_pos < len(obb_obj.ap) else np.nan

        per_class_rows.append({
            "class_id": cid_int,
            "class_name": cname,
            "OBB_P": round(obb_p, 4),
            "OBB_R": round(obb_r, 4),
            "OBB_F1": round(obb_f1, 4),
            "OBB_AP50": round(obb_ap50, 4),
            "OBB_AP50_95": round(obb_ap, 4),
        })

df_per_class = pd.DataFrame(per_class_rows)
print("\n=== MÉTRIQUES PAR CLASSE (OBB) ===")
display(df_per_class)

# 5. Matrice de confusion sous forme de tableau
df_cm = None
if getattr(metrics, "confusion_matrix", None) is not None:
    cm_mat = metrics.confusion_matrix.matrix
    labels_cm = [model.names[i] for i in sorted(model.names.keys())] + ["background"]
    if cm_mat.shape[0] == len(labels_cm):
        df_cm = pd.DataFrame(cm_mat.astype(int), index=[f"Pred_{l}" for l in labels_cm], columns=[f"True_{l}" for l in labels_cm])
    else:
        df_cm = pd.DataFrame(cm_mat.astype(int))
    print("\n=== MATRICE DE CONFUSION ===")
    display(df_cm)

# 6. Métriques par image (si disponibles dans cette version d'Ultralytics)
df_per_image = None
img_metrics = getattr(metrics, "image_metrics", None)
if isinstance(img_metrics, dict) and len(img_metrics) > 0:
    df_per_image = pd.DataFrame.from_dict(img_metrics, orient="index").reset_index().rename(columns={"index": "image"})
    print("\n=== MÉTRIQUES PAR IMAGE (aperçu) ===")
    display(df_per_image.head(20))

=== MÉTRIQUES GLOBALES (OBB) ===


,Metrique,Valeur
0,metrics/precision(B),0.885044
1,metrics/recall(B),0.529854
2,metrics/mAP50(B),0.619317
3,metrics/mAP50-95(B),0.477545
4,fitness,0.477545
5,OBB/Precision_moyenne (mp),0.885044
6,OBB/Recall_moyen (mr),0.529854
7,OBB/F1_moyen,0.662866
8,OBB/mAP50,0.619317
9,OBB/mAP75,0.539629



=== MÉTRIQUES PAR CLASSE (OBB) ===


,class_id,class_name,OBB_P,OBB_R,OBB_F1,OBB_AP50,OBB_AP50_95
0,0,cable,1.0000,0.0669,0.1254,0.2708,0.1033
1,1,cable-tie,0.5910,1.0000,0.7429,0.9950,0.8491
2,2,clip,1.0000,0.0000,0.0000,0.0831,0.0332
3,3,connector,0.8555,0.5824,0.6930,0.7526,0.6060
4,4,label,0.9787,1.0000,0.9893,0.9950,0.7960



=== MATRICE DE CONFUSION ===


,True_cable,True_cable-tie,True_clip,True_connector,True_label,True_background
Pred_cable,1,0,0,0,0,0
Pred_cable-tie,0,2,0,0,0,2
Pred_clip,0,0,0,0,0,0
Pred_connector,0,0,0,40,0,14
Pred_label,0,0,0,0,4,0
Pred_background,15,0,5,21,0,0


## Bloc 4 — Visualisation des courbes de validation (OBB) & Export des rapports

In [4]:
# 1. Export CSV et JSON complets
df_global.to_csv(save_dir / "metrics_globales.csv", index=False, encoding="utf-8-sig")
if not df_per_class.empty:
    df_per_class.to_csv(save_dir / "metrics_par_classe.csv", index=False, encoding="utf-8-sig")
if df_cm is not None:
    df_cm.to_csv(save_dir / "confusion_matrix.csv", encoding="utf-8-sig")
if df_per_image is not None:
    df_per_image.to_csv(save_dir / "metrics_par_image.csv", index=False, encoding="utf-8-sig")

# Export natif Ultralytics (CSV / JSON) si disponible
if hasattr(metrics, "to_csv"):
    (save_dir / "ultralytics_summary.csv").write_text(metrics.to_csv(), encoding="utf-8")
if hasattr(metrics, "to_json"):
    (save_dir / "ultralytics_summary.json").write_text(metrics.to_json(), encoding="utf-8")

print(f"Tous les tableaux CSV/JSON ont été exportés dans : {save_dir}")

# 2. Affichage des courbes et graphiques générés par Ultralytics
plot_files = sorted(save_dir.glob("*.png"))
if plot_files:
    n_plots = len(plot_files)
    cols = 2
    rows = (n_plots + 1) // 2
    fig, axes = plt.subplots(rows, cols, figsize=(16, 6 * rows))
    axes = np.array(axes).reshape(-1)
    for i, img_p in enumerate(plot_files):
        im = cv2.imread(str(img_p))
        axes[i].imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        axes[i].set_title(img_p.name)
        axes[i].axis("off")
    for j in range(i + 1, len(axes)):
        axes[j].axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Aucune image .png trouvée dans", save_dir)

Tous les tableaux CSV/JSON ont été exportés dans : D:\S5\Projet_d_expertise\projet\evaluations\03_orientation_des_composants_best_ep100_val


<Figure size 1600x1800 with 6 Axes>

## Bloc 5 — Benchmark comparatif des paliers (`best_ep10.pt` à `best_ep100.pt`)

In [5]:
CHECKPOINTS_DIR = Path(r"D:\S5\Projet_d_expertise\projet\models\03_orientation_des_composants_paliers").resolve()

if CHECKPOINTS_DIR.exists():
    ckpt_files = sorted(
        CHECKPOINTS_DIR.glob("best_ep*.pt"),
        key=lambda p: int("".join(filter(str.isdigit, p.stem)) or 0),
    )
    benchmark_rows = []
    for ckpt in ckpt_files:
        m_ckpt = YOLO(str(ckpt))
        res = m_ckpt.val(
            data=str(DATA_YAML),
            task="obb",
            split=EVAL_SPLIT,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            iou=IOU_THRESHOLD,
            device=DEVICE,
            plots=False,
            verbose=False,
        )
        m_obj = getattr(res, "obb", None) or getattr(res, "box", None)
        p_val = float(m_obj.mp)
        r_val = float(m_obj.mr)
        f1_val = (2 * p_val * r_val / (p_val + r_val)) if (p_val + r_val) > 0 else 0.0
        benchmark_rows.append({
            "Checkpoint": ckpt.name,
            "OBB_P": round(p_val, 4),
            "OBB_R": round(r_val, 4),
            "OBB_F1": round(f1_val, 4),
            "OBB_mAP50": round(float(m_obj.map50), 4),
            "OBB_mAP50_95": round(float(m_obj.map), 4),
        })

    if benchmark_rows:
        df_bench = pd.DataFrame(benchmark_rows)
        df_bench.to_csv(save_dir / f"benchmark_paliers_{EVAL_SPLIT}.csv", index=False, encoding="utf-8-sig")
        print(f"=== COMPARAISON DES PALIERS OBB ({EVAL_SPLIT}) ===")
        display(df_bench)
else:
    print(f"Dossier des paliers non trouvé : {CHECKPOINTS_DIR}")

Ultralytics 8.4.26  Python-3.12.7 torch-2.11.0+cpu CPU (Intel Core i5-10210U 1.60GHz)
YOLO11n-obb summary (fused): 110 layers, 2,654,698 parameters, 0 gradients, 6.6 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 69.235.7 MB/s, size: 31.1 KB)
val: Scanning D:\S5\Projet_d_expertise\projet\data\processed\wiring harness connectors segmen.v1i.yolov8-obb\valid\labels.cache... 22 images, 4 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 22/22  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 2.9s/it 5.8s<13.8s
                   all         22         88      0.898      0.561      0.571      0.445
Speed: 3.7ms preprocess, 231.8ms inference, 0.0ms loss, 9.0ms postprocess per image
Ultralytics 8.4.26  Python-3.12.7 torch-2.11.0+cpu CPU (Intel Core i5-10210U 1.60GHz)
YOLO11n-obb summary (fused): 110 layers, 2,654,698 parameters, 0 gradients, 6.6 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 89.932.6 MB/s, size: 41.6 

,Checkpoint,OBB_P,OBB_R,OBB_F1,OBB_mAP50,OBB_mAP50_95
0,best_ep10.pt,0.8979,0.5615,0.6909,0.5706,0.4451
1,best_ep20.pt,0.8772,0.5446,0.6720,0.5737,0.4333
2,best_ep30.pt,0.8896,0.5598,0.6872,0.5952,0.4535
3,best_ep40.pt,0.9189,0.5501,0.6882,0.5887,0.4585
4,best_ep50.pt,0.8908,0.5371,0.6701,0.6044,0.4814
5,best_ep60.pt,0.9380,0.5195,0.6687,0.6165,0.4839
6,best_ep70.pt,0.9227,0.5361,0.6781,0.6177,0.4890
7,best_ep80.pt,0.8975,0.5379,0.6727,0.6100,0.4653
8,best_ep90.pt,0.6018,0.5989,0.6003,0.6242,0.4805
9,best_ep100.pt,0.8850,0.5299,0.6629,0.6193,0.4775
